In [1]:
import argparse
import os

import pandas as pd
import pickle
from tqdm import tqdm
from copy import deepcopy

import numpy as np
import torch
from torch import optim
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt

import sys

In [32]:
d1 = 5000
r = 5
d2 = 100

mat1 = np.random.rand(d1, r)
mat2 = np.random.rand(r, d2)
M = mat1 @ mat2

In [7]:
from data_utils import load_data_all

dataset = 'ml-1m'
M = load_data_all(dataset)
M = M.numpy()

ml-1m


In [33]:
def differential_private_initialization(Y, G, tau, sigma0, p):
    m, n = Y.shape

    def projection(V, alpha2):
        V_projected = np.copy(V)
        for i in range(V_projected.shape[0]):
            row_norm = np.linalg.norm(V_projected[i, :], 2)
            if row_norm > alpha2:
                V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
        return V_projected

    # Step 1: Projection
    Y = projection(Y, G)

    # Step 2: Communicate (receive)
    Y_i = Y  # All users send their data to the server

    # Step 4: Server calculates private covariance matrix A
    A = tau**2 * np.sum([np.outer(Y_i[i], Y_i[i]) for i in range(m)], axis=0) / p**2 + np.random.normal(0, sigma0, size=(n, n))
    #A = (A + A.T) / 2  # Ensure A is symmetric

    # Step 5: Obtain (V^0, Σ) by performing the rank r SVD of A
    _, S, Vt = np.linalg.svd(A, full_matrices=False)

    V0 = Vt[:r, :].T
    Sigma = np.diag(S[:r])
    V0_tilde = V0 * np.sqrt(S[:r])

    U0 = np.array([(tau * Y[i].T @ V0 * np.sqrt(S[:r])/p) for i in range(m)])

    return U0.T, V0_tilde

# 初始化参数
# 创建一个与matrix同样大小的掩码，初始化为False
mask = np.zeros_like(M, dtype=bool)

# 找到非0元素的位置
p = 0.75
non_zero_indices = np.nonzero(M)
for i, j in zip(non_zero_indices[0], non_zero_indices[1]):
    if np.random.rand() < p:
        mask[i, j] = True

m, n = M.shape

Y = M * mask   # 示例数据

tau = 0.1



In [5]:
def P_omega(X):
    return X * mask

def P_omegai(X, i):
    return X * mask[i]

In [34]:
import numpy as np

def differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2):
    """
    Ut -> U
    """

    U = U0
    V = V0_tilde
    Y_hat = np.array([V @ U[i] for i in range(m)])

    def P_omega(X):
        mask = Y == 0
        return X * mask
    
    def P_omegai(X, i):
        mask = (Y == 0 )[i]
        return X * mask

    
    def projection(X, alpha):
        norm = np.linalg.norm(X, ord=2, axis=None)
        return X if norm <= alpha else (alpha / norm) * X
    
    def projection_(V, alpha):

        V_projected = np.copy(V)
        for i in range(V_projected.shape[0]):
            row_norm = np.linalg.norm(V_projected[i, :], 2)
            if row_norm > alpha2:
                V_projected[i, :] = (alpha2 / row_norm) * V_projected[i, :]
        return V_projected
    
    def projection_1d(V, alpha2):

        row_norm = np.linalg.norm(V, 2)
        if row_norm > alpha2:
            V = (alpha2 / row_norm) * V
        return V

    for t in tqdm(range(T)):
        if t == 0:            
            # 用户端接收
            P_omega_Y_hat_minus_Y = P_omega(Y_hat - Y)

        # 服务器端计算
        P_omega_Y_hat_minus_Y = projection(P_omega_Y_hat_minus_Y, G)

        R_tilde = np.sum([U[i] @ U[i].T for i in range(m)], axis=0) - V.T  @ V + np.random.normal(0, sigma1, size=(r, r))
        V = V - (eta / p) * (P_omega_Y_hat_minus_Y.T @ U + np.random.normal(0, sigma2, size=(n, r))) + (eta / 2 ) * V @ R_tilde
        V = projection(V, alpha2)

        # 用户端更新
        for i in range(m):
            U[i] = U[i] - (eta / p) * P_omegai(Y_hat[i].T - Y[i].T, i) @ V - (eta / 2) * U[i].T @ R_tilde
            U[i] = projection_1d(U[i], alpha1)
            Y_hat[i] = V @ U[i].T


    return U, V

# 初始化参数
T = 30
alpha1 = 3
alpha2 = 3
G = alpha1 * alpha2
eta = 0.3
sigma0 = 0.1
sigma1 = 0.1
sigma2 = 0.1
U0, V0_tilde = differential_private_initialization(Y, G, tau, sigma0, p)
U0 = U0.T


print(U0.shape)
print(V0_tilde.shape)

#print("Initialized U0^T:\n", U0)

U_final, V_final = differential_private_low_rank_matrix_completion(U0, V0_tilde, Y, T, alpha1, alpha2, G, eta, sigma1, sigma2)
print("Final U:\n", U_final)
print("Final V:\n", V_final)


(5000, 5)
(100, 5)


100%|██████████| 30/30 [00:34<00:00,  1.14s/it]

Final U:
 [[-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]
 [-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]
 [-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]
 ...
 [-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]
 [-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]
 [-1.34164415 -1.34164093 -1.34164005 -1.34163881 -1.34163999]]
Final V:
 [[-0.12746075 -0.12745998 -0.12746129 -0.12746419 -0.12746189]
 [-0.14436493 -0.14436402 -0.14436356 -0.14436305 -0.14436288]
 [-0.15219831 -0.15219877 -0.15219803 -0.15219598 -0.15219681]
 [-0.14081416 -0.14081508 -0.14081501 -0.14081551 -0.14081434]
 [-0.12461252 -0.12460958 -0.12461049 -0.12461068 -0.12461214]
 [-0.09942131 -0.0994222  -0.0994194  -0.09942163 -0.09941863]
 [-0.14909399 -0.14909651 -0.14909563 -0.14909362 -0.14909736]
 [-0.14700224 -0.14700464 -0.1470053  -0.14700199 -0.14700557]
 [-0.08323412 -0.08323216 -0.08323452 -0.0832331  -0.08323509]
 [-0.09983046 -0.09983047 -0.

In [35]:
X_estimation = U_final @ V_final.T
(np.linalg.norm(P_omega(M - X_estimation))**2)/(np.sum(mask))

0.2973654269608276

In [36]:
MTM = M.T @ M
VTV = V_final @ V_final.T
print(np.linalg.norm(VTV-MTM) / np.linalg.norm(MTM))

0.9999896891952809


In [37]:
XTX = X_estimation.T @ X_estimation
print(np.linalg.norm(XTX))
print(np.linalg.norm(MTM))
print(np.linalg.norm(XTX-MTM) / np.linalg.norm(MTM))

404999.9999996056
842236.1057549432
0.5506870613737633


In [31]:
mask

array([[False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       ...,
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False],
       [False, False, False, ..., False, False, False]])

In [18]:
print((X_estimation * mask)[0])


[ 4.09122682e-01 -6.68617919e-02  0.00000000e+00  0.00000000e+00
  2.07032851e-03  0.00000000e+00 -0.00000000e+00  0.00000000e+00
  4.19475770e-02  9.18527068e-02  1.16668061e-01  0.00000000e+00
 -0.00000000e+00  6.33437259e-02  8.11721669e-03 -2.12042828e-02
  1.73281383e-02 -9.06565260e-03  0.00000000e+00  6.15299720e-02
 -4.16662892e-02  8.86015334e-02  0.00000000e+00  6.20003407e-02
 -0.00000000e+00  0.00000000e+00 -5.53354725e-02 -0.00000000e+00
  0.00000000e+00 -4.17244793e-02  0.00000000e+00  1.18925163e-01
  0.00000000e+00  0.00000000e+00 -4.52704125e-02  0.00000000e+00
  1.72762610e-01  7.74082831e-02  0.00000000e+00  3.15998102e-02
  0.00000000e+00 -0.00000000e+00 -0.00000000e+00  0.00000000e+00
 -1.00992643e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  5.49220841e-02  0.00000000e+00  1.09185741e-01
  0.00000000e+00 -1.46558771e-02 -0.00000000e+00 -0.00000000e+00
  3.85525721e-02  0.00000000e+00  7.80928111e-02  0.00000000e+00
  0.00000000e+00 -9.66923

In [19]:
print(M[0])

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 5. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 5. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 4. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
